# 03 训练后怎么用 infer 加载 adapter 并对比

今天只解决一个问题：

> 我训练出 adapter 后，怎么验证它真的被加载了？怎么和 base 回答做对比？

核心文件回到你已经读过的：

```text
scripts/infer.py
```

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/codex备份/qwen-local-lora-sft-dpo学习版")

print("项目根目录:", PROJECT_ROOT)

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=80):
    lines = read_text(rel).splitlines()
    print(f"--- {rel}:{start}-{min(end, len(lines))} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = [i for i, line in enumerate(lines, start=1) if keyword in line]
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        show_file(rel, max(1, hit-context), min(len(lines), hit+context))
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 0. 今天只看哪些真实项目文件？

| 类型 | 路径 | 用途 |
|---|---|---|
| 推理脚本 | `scripts/infer.py` | 加载 base 和可选 adapter |
| 推荐 adapter | `outputs/sft_lora_qwen05b_custom_v3_from_v1_patch` | 正式项目推荐 checkpoint |
| 练习 adapter | `outputs/learning_try_tiny_sft_lora` | 你自己小训练可能产生的 adapter |

In [ ]:
for rel in ["scripts/infer.py", "outputs/sft_lora_qwen05b_custom_v3_from_v1_patch", "outputs/learning_try_tiny_sft_lora"]:
    print(f"{rel:55s}", "OK" if (PROJECT_ROOT / rel).exists() else "MISSING")

## 1. infer.py 哪里接收 adapter_path？

In [ ]:
find_lines("scripts/infer.py", "--adapter_path", context=6)
find_lines("scripts/infer.py", "if args.adapter_path", context=10)
find_lines("scripts/infer.py", "PeftModel.from_pretrained", context=8)

这就是推理时 adapter 的入口：

```text
加载 base Qwen
  -> 如果有 --adapter_path
  -> PeftModel.from_pretrained(model, adapter_path)
  -> base + adapter 一起 generate
```

## 2. adapter 目录里应该有什么？

LoRA adapter 不是完整模型。关键文件通常是：

```text
adapter_config.json
adapter_model.safetensors
tokenizer_config.json
```

In [ ]:
for rel in ["outputs/sft_lora_qwen05b_custom_v3_from_v1_patch", "outputs/learning_try_tiny_sft_lora"]:
    p = PROJECT_ROOT / rel
    print("\n", rel)
    if p.exists():
        for child in p.iterdir():
            if child.name in ["adapter_config.json", "adapter_model.safetensors", "tokenizer_config.json", "README.md"]:
                print(" ", child.name, child.stat().st_size)
    else:
        print("  不存在。练习训练后才会出现。")

## 3. base vs adapter 的命令区别

base 推理：不传 `--adapter_path`。

adapter 推理：传 `--adapter_path outputs/...`。

In [ ]:
prompt = "请用一句话解释 LoRA。"
base_cmd = [sys.executable, "scripts/infer.py", "--prompt", prompt, "--temperature", "0", "--max_new_tokens", "120", "--local_files_only"]
adapter_cmd = base_cmd + ["--adapter_path", "outputs/sft_lora_qwen05b_custom_v3_from_v1_patch"]
print("base 命令:")
print(" ".join(base_cmd))
print("\nadapter 命令:")
print(" ".join(adapter_cmd))

## 4. 安全对比：默认不运行模型

如果你确认环境没问题，可以把 `RUN_COMPARE=True`。这只做推理，不训练，不写文件。

In [ ]:
RUN_COMPARE = False
if RUN_COMPARE:
    for name, cmd in [("base", base_cmd), ("adapter", adapter_cmd)]:
        print("\n===", name, "===")
        result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, encoding="utf-8", errors="replace")
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
else:
    print("默认不运行。你先理解 base 命令和 adapter 命令的差别。")

## 5. 如果你训练了自己的 tiny adapter，怎么换？

把 adapter 命令里的路径换成你的练习输出：

```text
outputs/learning_try_tiny_sft_lora
```

In [ ]:
tiny_adapter_cmd = base_cmd + ["--adapter_path", "outputs/learning_try_tiny_sft_lora"]
print(" ".join(tiny_adapter_cmd))

## 6. 这一步你要能讲什么

> adapter 训练完以后，不是单独运行 adapter，而是用 `infer.py` 先加载 Qwen base，再通过 `PeftModel.from_pretrained` 挂上 adapter。验证时同一个 prompt 分别跑 base 和 adapter，观察回答是否按训练数据方向变化。